# 04-01 LLM 下載與基礎推論（2026 版）

## 學習目標

1. 了解如何從 HuggingFace Hub 下載開源 LLM，不依賴本機硬路徑。
2. 掌握 2026 統一載入慣例：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`。
3. 理解 `pipeline` 與 `AutoModel` 兩種推論入口的差異與適用場景。
4. 使用 `tokenizer.apply_chat_template()` 組裝對話 prompt，取代手寫格式字串。
5. 實作多輪對話迴圈，正確管理 message history。

## 前置需求

- 完成環境安裝（下一個 cell 會鎖版本）。
- 具備 HuggingFace Hub 帳號（免費，用於下載 gated model）。
- GPU 建議 ≥ 8 GB VRAM；若無 GPU，模型會自動 offload 至 CPU（速度較慢）。

## 與相鄰 notebook 的銜接

- 上一個主題：`../README.md`（模組概覽）
- 下一個主題：`../02-lora-finetune/`（LoRA 微調）— 本 notebook 下載的模型即為微調起點。
- `apply_chat_template` 在後續所有微調 notebook 中扮演關鍵角色，務必理解其語意。

In [ ]:
# [環境鎖版本] 執行一次即可；Colab / JupyterHub 請取消註解
# !pip install -q \
#   "transformers>=4.46" \
#   "accelerate>=1.0" \
#   "bitsandbytes>=0.44" \
#   "safetensors>=0.4" \
#   "huggingface_hub>=0.24" \
#   "torch>=2.4"

import sys
print("Python:", sys.version)

import transformers, accelerate, torch
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("torch       :", torch.__version__)
print("CUDA 可用    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. 從 HuggingFace Hub 下載模型

### 為何不用 ModelScope / 本機硬路徑？

硬路徑（如 `d:/Pretrained_models/`）綁定特定機器，造成可攜性為零；ModelScope 對 HuggingFace 官方模型沒有加值，卻引入額外 SDK。

2026 統一做法：**直接用 HuggingFace Hub model ID**，快取位置由環境變數 `HF_HOME` 控制（預設 `~/.cache/huggingface`），可跨平台、跨環境重現。

```python
from huggingface_hub import snapshot_download
snapshot_download(repo_id="...")  # 快取至 HF_HOME
```

若身處防火牆環境，可設定：
```bash
export HF_ENDPOINT=https://hf-mirror.com  # 鏡像站
```

In [ ]:
import os
from pathlib import Path

# 可選：自訂快取目錄（建議設在大容量磁碟）
# os.environ["HF_HOME"] = "/path/to/large/disk/.cache/huggingface"

# 確認目前快取位置
hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
print("HF_HOME:", hf_home)
print("存在   :", hf_home.exists())

## 2. HuggingFace Hub 登入

部分模型（如 Llama 系列）需要接受授權條款後才能下載，需登入 Hub。

- `huggingface-cli login` 是 CLI 登入方式（推薦在本機終端機執行）。
- `notebook_login()` 是 Jupyter 互動式登入，效果相同。
- Token 儲存於 `~/.cache/huggingface/token`，下次不需重新登入。

`Taiwan-LLM-7B-v2.1-chat` 為公開模型，**不需登入即可下載**，本 cell 僅作示範。

In [ ]:
# 若需要存取 gated model（如 meta-llama/Llama-3.1-8B-Instruct），執行此 cell
# from huggingface_hub import notebook_login
# notebook_login()

# 驗證目前登入狀態（未登入時會顯示 None）
from huggingface_hub import whoami
try:
    info = whoami()
    print("已登入為:", info["name"])
except Exception:
    print("尚未登入（公開模型可直接下載，無需登入）")

## 3. 2026 統一載入慣例：AutoTokenizer + AutoModelForCausalLM

### 三個關鍵參數

| 參數 | 2026 寫法 | 理由 |
|---|---|---|
| 精度 | `torch_dtype=torch.bfloat16` | bf16 動態範圍與 fp32 相同，不像 fp16 容易數值溢位；A10G/H100 的 bf16 硬體加速比 fp16 更快 |
| 裝置 | `device_map='auto'` | `auto` 讓 `accelerate` 根據 VRAM 自動分配層到 GPU/CPU/disk，單卡不夠時無縫 offload |
| 格式 | `use_safetensors=True` | safetensors 無法執行任意程式碼（pickle 可以），載入速度快 2-5x，且支援 mmap |

### bf16 vs fp16 vs fp32

```
fp32: 1 sign + 8 exp + 23 mantissa = 動態範圍大、精度高、速度最慢
fp16: 1 sign + 5 exp + 10 mantissa = 動態範圍小（容易 NaN）、速度快
bf16: 1 sign + 8 exp + 7 mantissa = 動態範圍與 fp32 相同、精度略低於 fp16 但訓練更穩定
```

**推論與微調統一用 bf16**（Ampere+ GPU）；fp16 只在必要時（如舊卡 V100）退而求其次。

### device_map='auto' 的工作原理

`accelerate` 會讀取 `config.json` 推算每層記憶體需求，依序嘗試放入：
1. GPU VRAM（最快）
2. CPU RAM（較慢，走 PCIe）
3. 磁碟（最慢，臨時 offload）

7B 模型以 bf16 約需 14 GB VRAM；若 VRAM 不足，部分層自動 offload 至 CPU。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 2026 統一載入慣例
# VRAM 預估：Taiwan-LLM-7B-v2.1-chat bf16 約 14 GB
# 若 VRAM < 14 GB，可改用量化版（見後續 notebook）
MODEL_ID = "yentinglin/Taiwan-LLM-7B-v2.1-chat"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",          # 自動 GPU/CPU/disk offload
    torch_dtype=torch.bfloat16, # bf16：動態範圍大、訓練/推論穩定
    use_safetensors=True,       # 安全格式，避免 pickle 任意程式碼執行
)

print("模型精度:", next(model.parameters()).dtype)
print("裝置分配:")
if hasattr(model, 'hf_device_map'):
    for k, v in model.hf_device_map.items():
        print(f"  {k}: {v}")

## 4. 使用 pipeline 進行推論

`pipeline` 是 HuggingFace 提供的高階推論介面，將 tokenize → forward → decode 封裝為單一 API。

使用 `device_map='auto'` 與 `from_pretrained` 語意一致，可自動處理多 GPU 或 CPU offload 的情況。

### 生成參數說明

| 參數 | 說明 |
|---|---|
| `max_new_tokens` | 只計算新生成的 token 數，比 `max_length` 更直覺 |
| `do_sample=True` | 啟用隨機採樣（否則為貪婪解碼） |
| `temperature=0.7` | 降低熵，讓輸出更集中（0 = 確定性，1 = 原始分布） |
| `top_k=50` | 每步只從機率最高的 50 個 token 中採樣 |
| `top_p=0.95` | Nucleus sampling：累積機率超過 95% 後截斷 |

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",          # 自動 GPU/CPU/disk offload
)

# 驗證 pipeline 裝置分配
print("pipeline 模型裝置分配：")
if hasattr(pipe.model, 'hf_device_map'):
    for k, v in list(pipe.model.hf_device_map.items())[:5]:
        print(f"  {k}: {v}")
    print("  ...")

## 5. apply_chat_template：跨模型可攜的對話格式

### 為何不能手寫 prompt 格式？

不同模型有各自的對話格式（Llama、ChatGLM、Mistral 等），手寫格式字串有三個致命問題：
1. 換模型就要改 code
2. 訓練與推論格式可能不一致（造成 performance 下降）
3. 多模態訊息（image/audio token）無法套用

**2026 解法**：所有模型在 `tokenizer_config.json` 中宣告自己的 `chat_template`（Jinja2 模板），`apply_chat_template()` 會自動套用正確格式。這也是通往 `05-Multimodal` 的關鍵橋樑——多模態的 `processor.apply_chat_template()` 延伸自同一套機制。

```python
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,             # 回傳字串，讓 pipeline 自行 tokenize
    add_generation_prompt=True  # 在結尾加上 assistant 開頭標記，觸發生成
)
```

In [ ]:
# 範例 1：氣象知識問答
messages = [
    {"role": "system", "content": "你是一個人工智慧助理"},
    {"role": "user",   "content": "東北季風如何影響台灣氣候？"},
]

# apply_chat_template 自動套用模型專屬格式
prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print("=== 組裝後的 prompt ===")
print(repr(prompt[:300]), "...")

outputs = pipe(
    prompt,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
)
print("\n=== 模型輸出 ===")
print(outputs[0]["generated_text"])

In [ ]:
# 範例 2：更換 system prompt，同一套 code 不需修改格式
messages = [
    {"role": "system", "content": "你是一個唱跳歌手"},
    {"role": "user",   "content": "以蜂蜜牛奶創作一首歌？"},
]

prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
outputs = pipe(
    prompt,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
)
print(outputs[0]["generated_text"])

## 6. 多輪對話迴圈

### role 名稱規範

OpenAI 標準與 HuggingFace chat template 規範的合法 role 為：`system`、`user`、`assistant`（三選一）。`assistance` 是常見拼字錯誤，`apply_chat_template` 會略過或報錯，務必使用 `assistant`。

### 輸出解析策略

`pipeline` 回傳完整的 `prompt + 生成內容`，需從中截取 assistant 的回應。
建議做法：用 `full_text[len(prompt):]` slice，而非依賴模型專屬 EOS token 字串（如 `split("</s>")`），後者換模型就會失效。

In [ ]:
def generate_response(messages: list[dict]) -> str:
    """Generate assistant reply and return only the new text."""
    prompt = pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
    )
    full_text: str = outputs[0]["generated_text"]
    # 截取 assistant 新生成的部分，不依賴模型專屬 EOS token 字串
    return full_text[len(prompt):].strip()


# 初始化對話：設定 system 角色
messages: list[dict] = [
    {"role": "system", "content": "你是一個唱跳歌手"},
]

# 多輪對話迴圈（Jupyter 環境下 input() 可直接互動）
print("進入對話模式，輸入 '退出' 結束。")
while True:
    user_input = input("User: ")
    print("User:", user_input)

    if user_input.strip() == "退出":
        print("對話結束。")
        break

    # 將用戶訊息加入 history
    messages.append({"role": "user", "content": user_input})

    # 生成回應
    assistant_reply = generate_response(messages)
    print("Assistant:", assistant_reply)

    # 將 assistant 回應加入 history，role 必須為 'assistant'
    messages.append({"role": "assistant", "content": assistant_reply})

In [ ]:
# 檢視對話 history，確認 role 欄位正確
for i, msg in enumerate(messages):
    print(f"[{i}] role={msg['role']!r:12s} | content={msg['content'][:60]!r}")

## 7. 小結與練習

### 本 notebook 核心要點

1. **路徑可攜性**：以 HF Hub model ID 取代本機硬路徑；`HF_HOME` 環境變數控制快取位置。
2. **統一載入慣例**：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True` 是 2026 標準三件組。
3. **apply_chat_template**：跨模型可攜的單一對話格式抽象，無需針對每個模型手寫 prompt 格式字串。
4. **role 命名**：合法值為 `system`、`user`、`assistant`，拼字錯誤會造成 silent bug。
5. **輸出截取**：用 `full_text[len(prompt):]` 截取新生成內容，不依賴模型專屬 EOS token。

### 練習題

1. 將 `MODEL_ID` 換成 `"meta-llama/Llama-3.1-8B-Instruct"`（需 Hub 登入），觀察 `apply_chat_template` 輸出的格式差異。
2. 在 `generate_response` 加入 `max_new_tokens` 參數，讓呼叫端可控制輸出長度。
3. 為多輪對話加入 context window 管理：當 `messages` 超過 N 輪時，保留 system prompt 並刪除最舊的 user/assistant pair。
4. 探索 `do_sample=False`（貪婪解碼）與 `do_sample=True`（隨機採樣）在同一 prompt 下的輸出差異。

### 下一步

前往 `../02-lora-finetune/` 學習如何對本 notebook 下載的模型進行 LoRA 微調，以及 `BitsAndBytesConfig` 量化配置（4-bit / 8-bit / 16-bit 的選擇時機）。